In [ ]:
# KALMAN GOLD-20 TRIGGER CONTRACT FORENSICS v2.7 — ONE CELL / READ ONLY
# Gold labels are ONLY the original revalidation_data_ready=True rows.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
from pathlib import Path
import pandas as pd, numpy as np, itertools, json

ROOT=Path("/content/drive/MyDrive/US_ETF/model_lab_v1/results")
AUD=ROOT/"open_revalidation_v1/open_revalidation_trade_audit.parquet"
a=pd.read_parquet(AUD)
gold=a[a["revalidation_data_ready"].fillna(False).astype(bool)].copy()
print("[AUDIT]",a.shape,"[GOLD]",gold.shape)
if len(gold)!=20: raise RuntimeError(f"Expected frozen gold-20, got {len(gold)}")

policies=sorted({c[:-9] for c in a.columns if c.startswith("OPEN_") and c.endswith("__trigger")})
features=["position_return_prev_close","position_return_open","position_return_5m","position_return_15m",
          "overnight_gap_return","open_momentum_5m","open_momentum_15m","giveback_prev_close_to_5m"]
features=[c for c in features if c in gold.columns]
print("[POLICIES]",policies)
print("[FEATURES]",features)

# Print gold rows first: small enough to inspect directly.
show=["fold","symbol","entry_timestamp","weight","entry_score"]+features
for p in policies: show += [p+"__trigger",p+"__net_return",p+"__exit_price"]
show=[c for c in show if c in gold.columns]
print("\n[GOLD-20 FULL TABLE]\n",gold[show].to_string(index=False))

# Candidate thresholds include policy-name thresholds plus exact midpoints between gold observations.
atoms=[]
for c in features:
    x=pd.to_numeric(gold[c],errors="coerce")
    vals=np.sort(x.dropna().unique())
    mids=[(vals[i]+vals[i+1])/2 for i in range(len(vals)-1)]
    ts=sorted(set([0.0,-0.002,0.002]+list(vals)+mids))
    for t in ts:
        atoms.append((f"{c}<={t:.12g}",(x<=t).fillna(False)))
        atoms.append((f"{c}<{t:.12g}",(x<t).fillna(False)))
        atoms.append((f"{c}>={t:.12g}",(x>=t).fillna(False)))
        atoms.append((f"{c}>{t:.12g}",(x>t).fillna(False)))

def metric(y,p):
    y=np.asarray(y,bool); p=np.asarray(p,bool)
    fp=int((~y&p).sum()); fn=int((y&~p).sum())
    return fp+fn,fp,fn,int(p.sum())

print("\n[EXACT GOLD-20 PREDICATE SEARCH]")
solutions={}
for pol in policies:
    y=gold[pol+"__trigger"].fillna(False).astype(bool)
    print("\n###",pol,"actual_true=",int(y.sum()))
    if y.sum()==0:
        print("NO POSITIVE GOLD LABELS: predicate is NOT identifiable from gold-20. Do not infer an always-false rule.")
        solutions[pol]=[]
        continue
    ranked=[]
    for n,p in atoms:
        e,fp,fn,ntrue=metric(y,p); ranked.append((e,1,n,fp,fn,ntrue,p))
    seeds=sorted(ranked,key=lambda z:(z[0],z[1],abs(z[5]-int(y.sum()))))[:120]
    # two-atom AND/OR
    for i,z1 in enumerate(seeds):
        for z2 in seeds[i+1:]:
            for op in ("AND","OR"):
                p=(z1[6]&z2[6]) if op=="AND" else (z1[6]|z2[6])
                e,fp,fn,ntrue=metric(y,p)
                ranked.append((e,2,f"({z1[2]}) {op} ({z2[2]})",fp,fn,ntrue,p))
    ranked=sorted(ranked,key=lambda z:(z[0],z[1],len(z[2])))
    exact=[]
    seen=set()
    for z in ranked:
        if z[0]!=0: break
        if z[2] not in seen:
            exact.append(z); seen.add(z[2])
        if len(exact)>=20: break
    for z in ranked[:10]: print("err=",z[0],"complexity=",z[1],"fp=",z[3],"fn=",z[4],"candidate_true=",z[5],z[2])
    print("[EXACT COUNT SHOWN]",len(exact))
    for z in exact[:10]: print(" EXACT",z[2])
    solutions[pol]=[z[2] for z in exact]

# Boundary evidence: positive vs negative feature ranges.
print("\n[POSITIVE/NEGATIVE BOUNDARIES]")
for pol in policies:
    y=gold[pol+"__trigger"].fillna(False).astype(bool)
    print("\n",pol,"true=",int(y.sum()),"false=",int((~y).sum()))
    for c in features:
        x=pd.to_numeric(gold[c],errors="coerce")
        pos=x[y].dropna(); neg=x[~y].dropna()
        if len(pos):
            print(c,"POS[min,max]=",float(pos.min()),float(pos.max()),
                  "NEG[min,max]=", (float(neg.min()),float(neg.max())) if len(neg) else None)

# Exit/net contract on gold positives and baseline on gold negatives.
print("\n[EXIT + NET + NONTRIGGER CONTRACT]")
exit_map={
 "OPEN_NEG_0BP_0M":"open_0_price_iex","OPEN_NEG_20BP_0M":"open_0_price_iex","OPEN_FLIP_0M":"open_0_price_iex",
 "OPEN_NEG_5M_CONFIRM":"open_5_price_iex","OPEN_FLIP_5M_CONFIRM":"open_5_price_iex",
 "OPEN_GAP_5M_CONFIRM":"open_5_price_iex","OPEN_GIVEBACK_5M":"open_5_price_iex","OPEN_NEG_15M_CONFIRM":"open_15_price_iex"}
ret_map={"open_0_price_iex":"position_return_open","open_5_price_iex":"position_return_5m","open_15_price_iex":"position_return_15m"}
for pol in policies:
    y=gold[pol+"__trigger"].fillna(False).astype(bool)
    print("\n",pol,"trigger=",int(y.sum()))
    if y.any():
        ep=pd.to_numeric(gold.loc[y,pol+"__exit_price"],errors="coerce")
        px=pd.to_numeric(gold.loc[y,exit_map[pol]],errors="coerce")
        nr=pd.to_numeric(gold.loc[y,pol+"__net_return"],errors="coerce")
        rr=pd.to_numeric(gold.loc[y,ret_map[exit_map[pol]]],errors="coerce")*pd.to_numeric(gold.loc[y,"weight"],errors="coerce")-pd.to_numeric(gold.loc[y,"cost_proxy"],errors="coerce")
        print("trigger_exit_price_maxerr=",float((ep-px).abs().max()),"trigger_net_maxerr=",float((nr-rr).abs().max()))
    nt=~y
    nr=pd.to_numeric(gold.loc[nt,pol+"__net_return"],errors="coerce")
    hist=pd.to_numeric(gold.loc[nt,"net_return"],errors="coerce")
    print("nontrigger_vs_historical_fixed4_maxerr=",float((nr-hist).abs().max()) if len(nr) else None)

print("\n[DECISION]")
for p,v in solutions.items():
    if gold[p+"__trigger"].fillna(False).astype(bool).sum()==0:
        print(p,": UNIDENTIFIABLE (0 positives in gold-20)")
    elif v:
        print(p,": exact gold-20 predicates found; source semantics still required if multiple predicates fit.")
    else:
        print(p,": no exact <=2-atom predicate found.")
print("\nREAD ONLY — no canonical file modified.")
print("Do NOT expand to 888 until each positive policy has a uniquely justified predicate; zero-positive policies require original source/spec or must remain unvalidated.")
